In [4]:
!pip install huggingface_hub --quiet

import torch
import numpy as np, random, os, pathlib, collections
from huggingface_hub import snapshot_download

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/framesight'
os.makedirs(PROJECT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda


# FrameSight - Phase 1: Data

Stream a 10-class subset of UCF101 (~500 clips) onto Colab with remotezip, without
downloading the full 6.5 GB archive. Videos land on the local VM; Phase 2 will turn
them into features cached to Drive. See docs/phases/phase-1-data.md.


In [5]:
# The Google Cloud mirror used by the TF tutorial now returns 403 (locked down).
# We use a pre-made UCF101 subset hosted on Hugging Face instead (~214 MB, public, no token).
download_root = snapshot_download(
    repo_id="ambarishnarayan/UCF_101_Subset",
    repo_type="dataset",
)
print("downloaded to:", download_root)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

downloaded to: /root/.cache/huggingface/hub/datasets--ambarishnarayan--UCF_101_Subset/snapshots/a93931a4119e546113bd59e6eaeeb6d3c9e8dc40


In [6]:
# Report the real structure before trusting it: extensions, the subset root, splits, classes, counts.
extension_counts = collections.Counter()
for current_dir, _, files in os.walk(download_root):
    for name in files:
        extension_counts[pathlib.Path(name).suffix.lower()] += 1
print("file extensions present:", dict(extension_counts))

# locate the folder that directly contains train/ and test/ (and usually val/)
def find_subset_root(base):
    for current_dir, subdirs, _ in os.walk(base):
        if "train" in subdirs and "test" in subdirs:
            return current_dir
    return None

subset_root = find_subset_root(download_root)
print("subset root:", subset_root)

if subset_root:
    for split in ["train", "val", "test"]:
        split_dir = pathlib.Path(subset_root) / split
        if not split_dir.exists():
            print(f"{split}: (missing)"); continue
        classes = sorted(p.name for p in split_dir.iterdir() if p.is_dir())
        total = sum(len(list((split_dir / c).iterdir())) for c in classes)
        print(f"{split}: {total} files across {len(classes)} classes")
    print("\nclasses:", classes)
    # show a couple of sample file paths so we can see the exact naming/extension
    sample_class = classes[0]
    sample_files = list((pathlib.Path(subset_root) / 'train' / sample_class).iterdir())[:3]
    print("sample files:", [p.name for p in sample_files])

file extensions present: {'.gz': 1, '': 1}
subset root: None


In [7]:
import tarfile, gzip

archive_files = []
for current_dir, _, names in os.walk(download_root):
    for name in names:
        archive_files.append(pathlib.Path(current_dir) / name)

print("files in the download:")
for f in archive_files:
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

# try to read each as a tar archive and report its internal layout
for f in archive_files:
    try:
        with tarfile.open(f, "r:*") as tar:
            members = [m for m in tar.getnames()]
            exts = collections.Counter(pathlib.Path(m).suffix.lower() for m in members)
            print(f"\n{f.name}: TAR archive, {len(members)} members, extensions {dict(exts)}")
            print("first members:", members[:8])
    except Exception as e:
        print(f"\n{f.name}: not a tar ({type(e).__name__})")

files in the download:
  UCF101_subset.tar.gz  (214.2 MB)
  .gitattributes  (0.0 MB)

UCF101_subset.tar.gz: TAR archive, 507 members, extensions {'': 50, '.avi': 457}
first members: ['UCF101_subset', 'UCF101_subset/test', 'UCF101_subset/train', 'UCF101_subset/val', 'UCF101_subset/val/BenchPress', 'UCF101_subset/val/BreastStroke', 'UCF101_subset/val/CleanAndJerk', 'UCF101_subset/val/Diving']

.gitattributes: not a tar (ReadError)


In [8]:
import tarfile

archive_path = next(f for f in archive_files if f.name == "UCF101_subset.tar.gz")
EXTRACT_ROOT = "/content"
with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(EXTRACT_ROOT, filter="data")   # filter='data' is the safe extraction mode (Py 3.12+)

SUBSET_DIR = pathlib.Path(EXTRACT_ROOT) / "UCF101_subset"
print("extracted to:", SUBSET_DIR)

extracted to: /content/UCF101_subset


In [9]:
for split in ["train", "val", "test"]:
    split_dir = SUBSET_DIR / split
    if not split_dir.exists():
        print(f"{split}: (missing)"); continue
    classes = sorted(p.name for p in split_dir.iterdir() if p.is_dir())
    counts = {c: len(list((split_dir / c).glob("*.avi"))) for c in classes}
    print(f"{split}: {sum(counts.values())} videos across {len(classes)} classes")

print("\nclasses:", classes)
print("per-class train counts:", {c: len(list((SUBSET_DIR / 'train' / c).glob('*.avi'))) for c in classes})

train: 270 videos across 9 classes
val: 90 videos across 9 classes
test: 97 videos across 9 classes

classes: ['BenchPress', 'BreastStroke', 'CleanAndJerk', 'Diving', 'FrontCrawl', 'JumpingJack', 'Lunges', 'PullUps', 'PushUps']
per-class train counts: {'BenchPress': 30, 'BreastStroke': 30, 'CleanAndJerk': 30, 'Diving': 30, 'FrontCrawl': 30, 'JumpingJack': 30, 'Lunges': 30, 'PullUps': 30, 'PushUps': 30}
